# benchmark_proseq

CAPY vs ProCapNet on PRO-seq gene-body (single strand). Loads the per-fold predictions saved by `examples/proseq/evaluate.py` (run with `--save_predictions`) and writes comparison CSVs + figures.

Count metrics are reported on **positive** TSS peaks, **negative** Rogers regions, and the **combined** set (mirroring `benchmark_procap`); profile metrics are positive-only. This requires running `evaluate.py` for both the positive split (e.g. `--split test`) and the negative split (`--split neg_test`) per fold.

Set the per-model `timestamp`s in the config cell to the runs you want to compare (use the fine-tuned timestamps if you ran `--stage both`).

In [ ]:
from __future__ import annotations

from pathlib import Path
import csv
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.spatial.distance import jensenshannon
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "examples" / "proseq" / "proseq_file_config.py").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing examples/proseq/proseq_file_config.py")


REPO_ROOT = find_repo_root()
EXAMPLES_PROSEQ_DIR = REPO_ROOT / "examples" / "proseq"
for path in [REPO_ROOT, EXAMPLES_PROSEQ_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from proseq_file_config import ProSeqFilesConfig
from performance_metrics import count_corr_mse_r2

pd.set_option("display.max_columns", None)

In [ ]:
# -----------------------------
# User config
# -----------------------------
proj_dir = Path("/grid/koo/home/ykang/elongation/CAPYBARA/results/runs")
treatment = "GSM8306530_0m"
data_type = "proseq"
folds = ["fold1", "fold2", "fold3", "fold4", "fold5", "fold6", "fold7"]
rc_augmented = False           # must match how evaluate.py was run (--reverse_complement)
positive_split = "test"        # PRO-seq TSS peaks (positives)
negative_split = "neg_test"    # Rogers negative regions; evaluate.py --split neg_test

# One timestamp per model, reused across all folds. Update to the runs to compare.
model_specs = [
    {"label": "ProCapNet", "model_name": "procapnet", "timestamp": "REPLACE_ME", "color": "#1565C0"},
    {"label": "CAPY", "model_name": "capy", "timestamp": "REPLACE_ME", "color": "#D97706"},
]

# Per-split scatter colors (positives vs negatives), like benchmark_procap.
pos_color = "#D97706"
neg_color = "#9E9E9E"

outdir = REPO_ROOT / "examples" / "proseq" / "results" / "benchmark"
outdir.mkdir(parents=True, exist_ok=True)
outdir

In [ ]:
# -----------------------------
# General helpers
# -----------------------------
def read_yaml(path):
    with Path(path).open("r") as handle:
        return yaml.safe_load(handle)


model_specs_df = pd.DataFrame(model_specs).copy()


def model_name_join(model_df=None):
    model_df = model_specs_df if model_df is None else model_df
    return "_".join(model_df["model_name"].tolist())


def get_fold_configs(model_name, timestamp):
    return [
        ProSeqFilesConfig.create(
            proj_dir=proj_dir,
            treatment=treatment,
            fold=fold,
            model_name=model_name,
            data_type=data_type,
            timestamp=timestamp,
        )
        for fold in folds
    ]


def split_peak_path(cfg, split_name):
    if split_name == "train":
        return cfg.train_peak_path
    if split_name == "val":
        return cfg.val_peak_path
    if split_name == "test":
        return cfg.test_peak_path
    if split_name == "neg_train":
        return cfg.neg_train_path
    if split_name == "neg_val":
        return cfg.neg_val_path
    if split_name == "neg_test":
        return cfg.neg_test_path
    raise ValueError(f"Unsupported split: {split_name}")


def _eval_file(rc_suffix, name, split_name):
    return f"{treatment}_{name}{rc_suffix}_{split_name}.npy"


def count_paths(cfg, split_name):
    rc_suffix = "_rc" if rc_augmented else ""
    base = Path(cfg.eval_dir)
    return (
        base / _eval_file(rc_suffix, "log_true_counts", split_name),
        base / _eval_file(rc_suffix, "log_pred_counts", split_name),
    )


def pred_profile_path(cfg, split_name):
    rc_suffix = "_rc" if rc_augmented else ""
    return Path(cfg.eval_dir) / _eval_file(rc_suffix, "log_pred_profiles", split_name)


def true_profile_path(cfg, split_name):
    rc_suffix = "_rc" if rc_augmented else ""
    return Path(cfg.eval_dir) / _eval_file(rc_suffix, "true_profiles", split_name)


def load_count_pair(cfg, split_name):
    true_path, pred_path = count_paths(cfg, split_name)
    missing = [str(p) for p in [true_path, pred_path] if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing count prediction file(s):\n  " + "\n  ".join(missing))
    true_log = np.asarray(np.load(true_path), dtype=float).reshape(-1)
    pred_log = np.asarray(np.load(pred_path), dtype=float).reshape(-1)
    if true_log.shape != pred_log.shape:
        raise ValueError(f"Count shape mismatch {cfg.model_name} {cfg.fold} {split_name}: true={true_log.shape}, pred={pred_log.shape}")
    return true_log, pred_log


def count_metrics(true_log, pred_log):
    true_log = np.asarray(true_log, dtype=float).reshape(-1)
    pred_log = np.asarray(pred_log, dtype=float).reshape(-1)
    finite = np.isfinite(true_log) & np.isfinite(pred_log)
    if not np.any(finite):
        raise ValueError("No finite count pairs available for metrics.")
    true_metric = true_log[finite].reshape(-1, 1, 1)
    pred_metric = pred_log[finite].reshape(-1, 1, 1)
    pearson, spearman, _mse, r2 = count_corr_mse_r2(true_metric, pred_metric)
    return {"pearson": float(pearson[0]), "spearman": float(spearman[0]), "r2": float(r2[0]), "n": int(finite.sum())}


def raw_profile_metrics(reference_profiles, query_profiles):
    # No smoothing, flatten the (single) strand, scipy Jensen-Shannon distance base 2.
    if reference_profiles.shape != query_profiles.shape:
        raise ValueError(f"Profile shape mismatch: reference={reference_profiles.shape}, query={query_profiles.shape}")
    jsds, pears, spears = [], [], []
    for i, (reference_prof, query_prof) in enumerate(zip(reference_profiles, query_profiles)):
        reference_flat = np.asarray(reference_prof, dtype=np.float64).reshape(-1)
        query_flat = np.asarray(query_prof, dtype=np.float64).reshape(-1)
        if not np.isfinite(reference_flat).all() or not np.isfinite(query_flat).all():
            raise ValueError(f"Non-finite profile values at index {i}")
        reference_flat = np.clip(reference_flat, 0, None)
        query_flat = np.clip(query_flat, 0, None)
        jsds.append(np.nan if reference_flat.sum() == 0 or query_flat.sum() == 0 else jensenshannon(reference_flat, query_flat, base=2))
        pears.append(np.corrcoef(reference_flat, query_flat)[0, 1])
        spears.append(spearmanr(reference_flat, query_flat).correlation)
    return {
        "profile_jsd": np.asarray(jsds, dtype=float),
        "profile_pearson": np.asarray(pears, dtype=float),
        "profile_spearman": np.asarray(spears, dtype=float),
    }


def benchmark_profile_metrics(true_profiles, pred_log_profiles):
    return raw_profile_metrics(true_profiles, np.exp(pred_log_profiles))

In [ ]:
# -----------------------------
# Plot helpers
# -----------------------------
def reads_for_plot(true_log, pred_log, floor):
    measured = np.expm1(np.asarray(true_log, dtype=float).reshape(-1))
    predicted = np.exp(np.asarray(pred_log, dtype=float).reshape(-1))
    measured = np.where(measured > 0, measured, floor)
    predicted = np.where(predicted > 0, predicted, floor)
    finite = np.isfinite(measured) & np.isfinite(predicted) & (measured > 0) & (predicted > 0)
    return measured[finite], predicted[finite]


def count_limits(pairs, floor):
    vals = []
    for true_log, pred_log in pairs:
        measured, predicted = reads_for_plot(true_log, pred_log, floor=floor)
        vals.extend([measured, predicted])
    all_values = np.concatenate([v for v in vals if len(v)])
    lo = max(float(np.nanmin(all_values)) * 0.8, floor)
    hi = float(np.nanmax(all_values)) * 1.2
    return lo, hi


def annotate_metrics(ax, metrics, fontsize=6.5):
    text = (
        f"Pearson $r$ = {metrics['pearson']:.3f}\n"
        f"Spearman $\\rho$ = {metrics['spearman']:.3f}\n"
        f"R2 = {metrics['r2']:.3f}\n"
        f"N = {metrics['n']:,}"
    )
    ax.text(0.05, 0.95, text, transform=ax.transAxes, va="top", ha="left", fontsize=fontsize, linespacing=1.12)


def style_count_ax(ax, lim, title, show_ylabel):
    ax.plot(lim, lim, color="black", linewidth=0.6, linestyle="--", alpha=0.6)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title, fontsize=8)
    ax.set_xlabel("Measured Reads", fontsize=7)
    if show_ylabel:
        ax.set_ylabel("Predicted Reads", fontsize=7)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=2, labelsize=6)


def draw_single_count_panel(ax, true_log, pred_log, color, lim, title, show_ylabel, *, alpha, s, floor):
    measured, predicted = reads_for_plot(true_log, pred_log, floor=floor)
    ax.scatter(measured, predicted, alpha=alpha, s=s, color=color, linewidths=0)
    style_count_ax(ax, lim, title, show_ylabel)
    annotate_metrics(ax, count_metrics(true_log, pred_log))


def draw_combined_count_panel(
    ax,
    pos_true,
    pos_pred,
    neg_true,
    neg_pred,
    lim,
    title,
    show_ylabel,
    *,
    pos_alpha_value,
    neg_alpha_value,
    s,
    floor,
    pos_color_value,
    neg_color_value,
):
    # Negatives behind positives; annotation reports the combined (pos+neg) metrics.
    neg_measured, neg_predicted = reads_for_plot(neg_true, neg_pred, floor=floor)
    pos_measured, pos_predicted = reads_for_plot(pos_true, pos_pred, floor=floor)
    ax.scatter(neg_measured, neg_predicted, alpha=neg_alpha_value, s=s, color=neg_color_value, linewidths=0, label=f"Negative N={len(neg_true):,}")
    ax.scatter(pos_measured, pos_predicted, alpha=pos_alpha_value, s=s, color=pos_color_value, linewidths=0, label=f"Positive N={len(pos_true):,}")
    style_count_ax(ax, lim, title, show_ylabel)
    annotate_metrics(ax, count_metrics(np.r_[pos_true, neg_true], np.r_[pos_pred, neg_pred]))
    ax.legend(frameon=False, fontsize=5.5, loc="lower right", handlelength=1.0, borderaxespad=0.2)


def save_figure(fig, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight", pad_inches=0.08)
    plt.show()
    print("saved:", path)

In [ ]:
# -----------------------------
# Load observed single-strand positive-split profiles once, from the first model's
# saved npy (both models evaluate the identical positive test set, so truth is shared).
# Profile metrics are computed on positives only (negatives have ~no profile signal).
# -----------------------------
ref_cfgs = get_fold_configs(model_specs_df.iloc[0]["model_name"], model_specs_df.iloc[0]["timestamp"])
params = read_yaml(Path(ref_cfgs[0].params_path))
output_length = int(params["dataset"]["output_length"])

profile_truth_rows = []
true_profiles_all = []
fold_labels = []
for cfg in ref_cfgs:
    path = true_profile_path(cfg, positive_split)
    if not path.exists():
        raise FileNotFoundError(f"Missing observed-profile file (run evaluate.py with --save_predictions): {path}")
    profiles = np.asarray(np.load(path), dtype=float)
    profile_truth_rows.append({"fold": cfg.fold, "true_profiles": profiles, "n": int(profiles.shape[0])})
    true_profiles_all.append(profiles)
    fold_labels.extend([cfg.fold] * profiles.shape[0])

profile_truth_df = pd.DataFrame(profile_truth_rows)
true_profiles = np.concatenate(true_profiles_all, axis=0)
fold_labels = np.asarray(fold_labels)
print("true_profiles:", true_profiles.shape)
profile_truth_df[["fold", "n"]]

In [ ]:
# -----------------------------
# "Average profile across other folds" baseline (single strand)
# -----------------------------
fold_ids = list(dict.fromkeys(fold_labels.tolist()))
avg_profiles_by_fold = {}
for fold in fold_ids:
    other = fold_labels != fold
    if not np.any(other):
        raise ValueError(f"No other folds to build the average-profile baseline for {fold}")
    avg_profiles_by_fold[fold] = np.mean(true_profiles[other], axis=0)
avg_profiles_over_folds = np.stack([avg_profiles_by_fold[fold] for fold in fold_labels], axis=0)
avg_profile_metrics = raw_profile_metrics(true_profiles, avg_profiles_over_folds)
print("avg-baseline profile JSD mean:", float(np.nanmean(avg_profile_metrics["profile_jsd"])))

In [ ]:
# -----------------------------
# Load count/profile predictions for each model; compute per-example profile metrics.
# Counts are loaded for BOTH the positive and negative splits (tagged via "split");
# profiles are positive-only.
# -----------------------------
count_rows = []
profile_pred_rows = []
profile_metric_rows = []

for _, spec in model_specs_df.iterrows():
    label = spec["label"]
    fold_cfgs = get_fold_configs(spec["model_name"], spec["timestamp"])
    model_params = read_yaml(Path(fold_cfgs[0].params_path))
    model_output_length = int(model_params["dataset"]["output_length"])
    if model_output_length != output_length:
        raise ValueError(f"{label} output_length={model_output_length}, expected {output_length}")

    pred_log_profiles_by_fold = []
    for cfg in fold_cfgs:
        fold = cfg.fold
        true_profiles_fold = profile_truth_df.loc[profile_truth_df["fold"] == fold, "true_profiles"].iloc[0]
        for split_label, split_name in [("pos", positive_split), ("neg", negative_split)]:
            true_log, pred_log = load_count_pair(cfg, split_name)
            count_rows.append({"model": label, "model_name": spec["model_name"], "timestamp": spec["timestamp"], "color": spec["color"], "fold": fold, "split": split_label, "true_log": true_log, "pred_log": pred_log, "n": int(true_log.size)})

        prof_path = pred_profile_path(cfg, positive_split)
        if not prof_path.exists():
            raise FileNotFoundError(f"Missing profile prediction file: {prof_path}")
        pred_log_profiles_fold = np.asarray(np.load(prof_path), dtype=float)
        if pred_log_profiles_fold.shape != true_profiles_fold.shape:
            raise ValueError(f"Profile shape mismatch for {label} {fold}: pred={pred_log_profiles_fold.shape}, true={true_profiles_fold.shape}")
        profile_pred_rows.append({"model": label, "model_name": spec["model_name"], "timestamp": spec["timestamp"], "color": spec["color"], "fold": fold, "pred_log_profiles": pred_log_profiles_fold, "n": int(pred_log_profiles_fold.shape[0])})
        pred_log_profiles_by_fold.append(pred_log_profiles_fold)

    pred_log_profiles = np.concatenate(pred_log_profiles_by_fold, axis=0)
    metrics = benchmark_profile_metrics(true_profiles, pred_log_profiles)
    for metric_name, values in metrics.items():
        for fold in folds:
            mask = fold_labels == fold
            profile_metric_rows.append({"model": label, "model_name": spec["model_name"], "timestamp": spec["timestamp"], "color": spec["color"], "fold": fold, "metric": metric_name, "values": values[mask], "mean": float(np.nanmean(values[mask])), "n": int(mask.sum())})

count_df = pd.DataFrame(count_rows)
profile_pred_df = pd.DataFrame(profile_pred_rows)
profile_metric_df = pd.DataFrame(profile_metric_rows)
print("count_df rows:", len(count_df), "profile_pred_df rows:", len(profile_pred_df), "profile_metric_df rows:", len(profile_metric_df))
count_df[["model", "fold", "split", "n"]].head()

In [ ]:
# -----------------------------
# Dataframe accessors
# -----------------------------
def count_row(model, fold, split="pos"):
    rows = count_df[(count_df["model"] == model) & (count_df["fold"] == fold) & (count_df["split"] == split)]
    if len(rows) != 1:
        raise ValueError(f"Expected one count row for model={model}, fold={fold}, split={split}; got {len(rows)}")
    return rows.iloc[0]


def count_arrays(model, fold, split="pos"):
    # Combined "pos_neg" stacks the positive and negative arrays for that fold.
    if split == "pos_neg":
        pos = count_row(model, fold, "pos")
        neg = count_row(model, fold, "neg")
        return np.r_[pos["true_log"], neg["true_log"]], np.r_[pos["pred_log"], neg["pred_log"]]
    row = count_row(model, fold, split)
    return row["true_log"], row["pred_log"]


def aggregate_count_arrays(model, split="pos"):
    trues, preds = [], []
    for fold in folds:
        true_log, pred_log = count_arrays(model, fold, split)
        trues.append(true_log)
        preds.append(pred_log)
    return np.concatenate(trues), np.concatenate(preds)


def model_color(model):
    return model_specs_df.loc[model_specs_df["label"] == model, "color"].iloc[0]

In [ ]:
# -----------------------------
# Numerical results CSVs
# -----------------------------
def write_csv(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = sorted({key for row in rows for key in row.keys()})
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print("saved:", path)


def fold_mean_std(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan"), float("nan")
    if values.size == 1:
        return float(values[0]), float("nan")
    return float(np.mean(values)), float(np.std(values, ddof=1))


# Counts are summarised per split: positives, negatives, and positive+negative combined.
count_splits = ["pos", "neg", "pos_neg"]
count_fold_rows = []
count_aggregate_rows = []
profile_fold_rows = []
profile_aggregate_rows = []

for _, spec in model_specs_df.iterrows():
    label = spec["label"]
    for split in count_splits:
        for fold in folds:
            true_log, pred_log = count_arrays(label, fold, split)
            m = count_metrics(true_log, pred_log)
            count_fold_rows.append({"model": label, "model_name": spec["model_name"], "timestamp": spec["timestamp"], "split": split, "fold": fold, "pearson": m["pearson"], "spearman": m["spearman"], "r2": m["r2"], "n": m["n"]})

        true_log, pred_log = aggregate_count_arrays(label, split)
        m = count_metrics(true_log, pred_log)
        split_fold_rows = [r for r in count_fold_rows if r["model"] == label and r["split"] == split]
        pearson_mean, pearson_std = fold_mean_std([r["pearson"] for r in split_fold_rows])
        spearman_mean, spearman_std = fold_mean_std([r["spearman"] for r in split_fold_rows])
        r2_mean, r2_std = fold_mean_std([r["r2"] for r in split_fold_rows])
        count_aggregate_rows.append({"model": label, "model_name": spec["model_name"], "timestamp": spec["timestamp"], "split": split, "pearson": m["pearson"], "spearman": m["spearman"], "r2": m["r2"], "pearson_fold_mean": pearson_mean, "pearson_fold_std": pearson_std, "spearman_fold_mean": spearman_mean, "spearman_fold_std": spearman_std, "r2_fold_mean": r2_mean, "r2_fold_std": r2_std, "n": m["n"]})

    for fold in folds:
        sub = profile_metric_df[(profile_metric_df["model"] == label) & (profile_metric_df["fold"] == fold)]
        profile_fold_rows.append({"model": label, "model_name": spec["model_name"], "timestamp": spec["timestamp"], "fold": fold, "profile_jsd_mean": float(sub.loc[sub["metric"] == "profile_jsd", "mean"].iloc[0]), "profile_pearson_mean": float(sub.loc[sub["metric"] == "profile_pearson", "mean"].iloc[0]), "profile_spearman_mean": float(sub.loc[sub["metric"] == "profile_spearman", "mean"].iloc[0]), "n": int(sub["n"].iloc[0])})

    model_profile = profile_metric_df[profile_metric_df["model"] == label]
    model_profile_fold_rows = [r for r in profile_fold_rows if r["model"] == label]
    pj_mean, pj_std = fold_mean_std([r["profile_jsd_mean"] for r in model_profile_fold_rows])
    pp_mean, pp_std = fold_mean_std([r["profile_pearson_mean"] for r in model_profile_fold_rows])
    ps_mean, ps_std = fold_mean_std([r["profile_spearman_mean"] for r in model_profile_fold_rows])
    profile_aggregate_rows.append({"model": label, "model_name": spec["model_name"], "timestamp": spec["timestamp"], "profile_jsd_mean": float(np.nanmean(np.concatenate(model_profile.loc[model_profile["metric"] == "profile_jsd", "values"].to_list()))), "profile_pearson_mean": float(np.nanmean(np.concatenate(model_profile.loc[model_profile["metric"] == "profile_pearson", "values"].to_list()))), "profile_spearman_mean": float(np.nanmean(np.concatenate(model_profile.loc[model_profile["metric"] == "profile_spearman", "values"].to_list()))), "profile_jsd_fold_mean": pj_mean, "profile_jsd_fold_std": pj_std, "profile_pearson_fold_mean": pp_mean, "profile_pearson_fold_std": pp_std, "profile_spearman_fold_mean": ps_mean, "profile_spearman_fold_std": ps_std, "n": int(sum(model_profile.loc[model_profile["metric"] == "profile_jsd", "n"]))})

write_csv(outdir / "count_metrics_by_fold.csv", count_fold_rows)
write_csv(outdir / "count_metrics_aggregate.csv", count_aggregate_rows)
write_csv(outdir / "profile_metrics_by_fold.csv", profile_fold_rows)
write_csv(outdir / "profile_metrics_aggregate.csv", profile_aggregate_rows)

In [ ]:
count_metrics_df = pd.DataFrame(count_fold_rows)
for model_name in count_metrics_df["model_name"].unique():
    for split in count_splits:
        model_df = count_metrics_df[(count_metrics_df["model_name"] == model_name) & (count_metrics_df["split"] == split)]
        print(f"=== {model_name} [{split}] (count, mean +/- std across folds) ===")
        for metric in ["pearson", "spearman", "r2"]:
            print(f"  {metric}: {model_df[metric].mean():.4f} +/- {model_df[metric].std():.4f}")

In [ ]:
profile_fold_summary_df = pd.DataFrame(profile_fold_rows)
for model_name in profile_fold_summary_df["model_name"].unique():
    model_df = profile_fold_summary_df[profile_fold_summary_df["model_name"] == model_name]
    print(f"=== {model_name} (profile, mean +/- std across folds) ===")
    for metric in ["profile_jsd_mean", "profile_pearson_mean", "profile_spearman_mean"]:
        print(f"  {metric}: {model_df[metric].mean():.4f} +/- {model_df[metric].std():.4f}")

In [ ]:
# -----------------------------
# Count scatter plots (measured vs predicted reads), positives over negatives
# -----------------------------
plot_floor = 0.5
joined = model_name_join()

# Per-fold, one row per model (each panel overlays that fold's positives + negatives)
for _, spec in model_specs_df.iterrows():
    label = spec["label"]
    fold_pairs = [count_arrays(label, f, "pos_neg") for f in folds]
    lim = count_limits(fold_pairs, floor=plot_floor)
    fig, axes = plt.subplots(1, len(folds), figsize=(2.25 * len(folds), 2.25), dpi=200, sharex=True, sharey=True)
    for i, (ax, fold) in enumerate(zip(np.atleast_1d(axes), folds)):
        pos = count_row(label, fold, "pos")
        neg = count_row(label, fold, "neg")
        draw_combined_count_panel(ax, pos["true_log"], pos["pred_log"], neg["true_log"], neg["pred_log"], lim, f"{label} {fold}", i == 0, pos_alpha_value=0.18, neg_alpha_value=0.10, s=2, floor=plot_floor, pos_color_value=pos_color, neg_color_value=neg_color)
    save_figure(fig, outdir / f"count_scatter_folds_{spec['model_name']}.pdf")

# Aggregated across folds, one column per model
lim = count_limits([aggregate_count_arrays(label, "pos_neg") for label in model_specs_df["label"]], floor=plot_floor)
fig, axes = plt.subplots(1, len(model_specs_df), figsize=(2.8 * len(model_specs_df), 2.8), dpi=240, sharex=True, sharey=True)
for i, (ax, (_, spec)) in enumerate(zip(np.atleast_1d(axes), model_specs_df.iterrows())):
    pos_true, pos_pred = aggregate_count_arrays(spec["label"], "pos")
    neg_true, neg_pred = aggregate_count_arrays(spec["label"], "neg")
    draw_combined_count_panel(ax, pos_true, pos_pred, neg_true, neg_pred, lim, spec["label"], i == 0, pos_alpha_value=0.18, neg_alpha_value=0.10, s=1, floor=plot_floor, pos_color_value=pos_color, neg_color_value=neg_color)
save_figure(fig, outdir / f"count_scatter_aggregate_{joined}.pdf")

In [ ]:
# -----------------------------
# Profile JSD cumulative distribution over test peaks
# -----------------------------
fig, ax = plt.subplots(figsize=(3.0, 2.1), dpi=200)

avg_jsd = np.asarray(avg_profile_metrics["profile_jsd"], dtype=float)
avg_values = np.sort(avg_jsd[np.isfinite(avg_jsd)])
ax.plot(avg_values, np.arange(1, len(avg_values) + 1) / len(avg_values), color="#A0CB95", linewidth=1.2, label="Avg. profile vs. measured")

for _, spec in model_specs_df.iterrows():
    pred_log_profiles = np.concatenate(profile_pred_df.loc[profile_pred_df["model"] == spec["label"]].sort_values("fold")["pred_log_profiles"].to_list(), axis=0)
    values = benchmark_profile_metrics(true_profiles, pred_log_profiles)["profile_jsd"]
    values = np.sort(values[np.isfinite(values)])
    ax.plot(values, np.arange(1, len(values) + 1) / len(values), color=spec["color"], linewidth=1.2, label=f"{spec['label']} vs. measured")
ax.set_xlabel("Jensen-Shannon Distance", fontsize=8)
ax.set_ylabel("Cum. frac.\ntest peaks", fontsize=8)
ax.set_xlim(0, 1.01)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(length=2, labelsize=7)
ax.legend(frameon=False, fontsize=6, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=max(1, len(model_specs_df)))
fig.subplots_adjust(left=0.18, right=0.98, bottom=0.23, top=0.74)
save_figure(fig, outdir / "cmp_cdf_profile_jsd.pdf")

In [ ]:
# -----------------------------
# Fold-level profile/count metric summaries (positives vs positive+negative counts)
# -----------------------------
metric_grid = [
    ("profile_jsd", "count_pos_r2", "count_pos_neg_r2"),
    ("profile_pearson", "count_pos_pearson", "count_pos_neg_pearson"),
    ("profile_spearman", "count_pos_spearman", "count_pos_neg_spearman"),
]
metric_titles = {
    "profile_jsd": "Profile JSD (lower better)",
    "profile_pearson": "Profile Pearson",
    "profile_spearman": "Profile Spearman",
    "count_pos_r2": "Count R2\n(Positive)",
    "count_pos_pearson": "Count Pearson\n(Positive)",
    "count_pos_spearman": "Count Spearman\n(Positive)",
    "count_pos_neg_r2": "Count R2\n(Positive + Negative)",
    "count_pos_neg_pearson": "Count Pearson\n(Positive + Negative)",
    "count_pos_neg_spearman": "Count Spearman\n(Positive + Negative)",
}

fold_values = {}
for _, spec in model_specs_df.iterrows():
    label = spec["label"]
    for _, prow in profile_pred_df[profile_pred_df["model"] == label].sort_values("fold").iterrows():
        fold = prow["fold"]
        tf = profile_truth_df.loc[profile_truth_df["fold"] == fold, "true_profiles"].iloc[0]
        m = benchmark_profile_metrics(tf, prow["pred_log_profiles"])
        for name in ["profile_jsd", "profile_pearson", "profile_spearman"]:
            fold_values.setdefault((label, name), {})[fold] = float(np.nanmean(m[name]))
    for name, split, source in [
        ("count_pos_r2", "pos", "r2"),
        ("count_pos_pearson", "pos", "pearson"),
        ("count_pos_spearman", "pos", "spearman"),
        ("count_pos_neg_r2", "pos_neg", "r2"),
        ("count_pos_neg_pearson", "pos_neg", "pearson"),
        ("count_pos_neg_spearman", "pos_neg", "spearman"),
    ]:
        fold_values.setdefault((label, name), {})
        for fold in folds:
            true_log, pred_log = count_arrays(label, fold, split)
            mm = count_metrics(true_log, pred_log)
            fold_values[(label, name)][fold] = float(mm[source])

x = np.arange(len(folds))
fig, axes = plt.subplots(len(metric_grid), 3, figsize=(10.5, 6.0), dpi=200, sharex=True)
for r, metric_row in enumerate(metric_grid):
    for c, metric in enumerate(metric_row):
        ax = axes[r, c]
        for _, spec in model_specs_df.iterrows():
            vals = fold_values[(spec["label"], metric)]
            ys = [vals[f] for f in folds]
            ax.plot(x, ys, marker="o", linewidth=1.2, markersize=2.5, color=spec["color"], label=spec["label"])
        ax.set_title(metric_titles[metric], fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(folds, rotation=45, fontsize=6)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(length=2, labelsize=7)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, fontsize=8, ncol=len(labels), loc="upper center", bbox_to_anchor=(0.5, 1.03))
fig.tight_layout(rect=(0, 0, 1, 0.95))
save_figure(fig, outdir / "cmp_fold_metrics.pdf")